# SRO 분석 — 단거리 규칙성 (Short-Range Order)

비정질 구조의 **단거리 규칙성**을 트래젝토리 시간평균으로 분석합니다.
원소 비종속(`amorph` 패키지) — `element`/`type` 컬럼에서 원소를 자동 감지합니다.

분석 항목: g(r) · cutoff 비교 · R(r)/G(r) · CN · ADF · CSRO · Voronoi · BOO · 혼성화 · 사면체 질서

**사용법**: 아래 *Config* 셀의 값만 바꾸고 위에서부터 순서대로 실행하세요.

## 0. Import

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import amorph
from amorph import core
from amorph.sro import rdf, coordination, csro, voronoi, boo, hybridization, tetrahedra
from amorph.core import cutoffs as cut
from amorph import presets
from amorph.core.elements import element_colors
print('amorph', amorph.__version__)

## 1. Config — 여기 값만 바꾼다

- `TRAJ` : dump 파일 경로 (예: `dump.300K.lammpstrj`, `dump_T0300.lammpstrj`).
  데이터 파일(`.data`)을 넣어도 됩니다 (단일 프레임 폴백).
- `TYPE_MAP` : dump에 `element` 컬럼이 없을 때만 사용 (atom-style dump). 있으면 무시됨.
- `FIXED_CUTOFFS` : 결합 cutoff(Å). SiCN 기본값은 LAMMPS 입력과 동일. C-N·N-N=0(비결합).
- `PROD_RANGE`, `STRIDE` : 시간평균에 쓸 production 프레임 구간/간격.
- 정확성을 위해 frame이 많을수록 좋습니다 (block 에러로 ±1σ 산출).

In [ ]:
TRAJ        = 'dump.300K.lammpstrj'      # ← 분석할 trajectory/data 파일
TYPE_MAP    = presets.SICN_TYPE_MAP       # {1:'Si',2:'C',3:'N'}  (element 없을 때만)
FIXED_CUTOFFS = presets.SICN_CUTOFFS      # 임의 시스템이면 직접 {('A','B'):rc} 작성

PROD_RANGE  = None        # (lo, hi) 프레임 구간. None=전체. 예: (50, None)
STRIDE      = 1           # decorrelation 간격
R_MAX       = 10.0        # g(r) 최대 거리 (Å) — LAMMPS rdf cutoff과 동일
NBINS       = 500         # g(r) bin 수
NBLOCKS     = 5           # block 평균 개수 (±1σ 에러)
BOO_K       = 4           # BOO 이웃 수 (사면체=4)

## 2. Load — 트래젝토리 읽기 + 시스템 정보

원소·조성·밀도는 프레임 평균으로 보고합니다 (NPT라 부피가 변하므로).

In [ ]:
frames_all = core.load(TRAJ, type_map=TYPE_MAP, frames='all')
traj = core.select_frames(frames_all, frame_range=PROD_RANGE, stride=STRIDE)
species = core.species_of(traj)
pairs = core.unique_pairs(species)
CM = core.CutoffMatrix(FIXED_CUTOFFS, default=0.0)
colors = element_colors(species)

print(f'총 {len(frames_all)} 프레임 중 production {len(traj)} 프레임 사용')
print(f'원소: {species}   원자수: {traj[0].n_atoms}')
comp = np.mean([[fr.count(e)/fr.n_atoms for e in species] for fr in traj], axis=0)
rho  = np.mean([fr.mass_density() for fr in traj])
nrho = np.mean([fr.number_density() for fr in traj])
print('조성(mol%):', {e: f'{c*100:.2f}' for e,c in zip(species, comp)})
print(f'질량밀도 = {rho:.4f} g/cc    수밀도 = {nrho:.5f} atoms/Å³')
print('cutoff 행렬:', CM)

## 3. partial + total g(r)  (시간평균 ±1σ)

In [ ]:
R = rdf.partial_rdf(traj, pairs=pairs, r_max=R_MAX, nbins=NBINS, n_blocks=NBLOCKS)

fig, ax = plt.subplots(figsize=(9,5.5))
for (A,B) in pairs:
    g, e = R[(A,B)]['g'], R[(A,B)]['err']
    line, = ax.plot(R['r'], g, lw=1.3, label=f'{A}-{B}')
    ax.fill_between(R['r'], g-e, g+e, alpha=0.2, color=line.get_color())
ax.plot(R['r'], R['total']['g'], 'k-', lw=1.8, label='total', alpha=0.8)
ax.axhline(1.0, color='gray', lw=0.6, ls=':')
ax.set_xlabel('r (Å)'); ax.set_ylabel('g(r)'); ax.set_xlim(0, R_MAX)
ax.set_title('Partial + total radial distribution g(r)')
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3); plt.show()

## 4. g(r) 봉우리 측정 — 위치 · 높이 · FWHM · Gaussian 적합(R²,RMSE)

In [ ]:
print(f"{'pair':<8}{'peak_r':>9}{'height':>9}{'FWHM':>9}{'R^2':>8}{'RMSE':>9}")
for (A,B) in pairs:
    if CM.get(A,B) <= 0:  # 비결합 쌍은 첫 피크 의미 없음
        continue
    m = rdf.measure_peaks(R['r'], R[(A,B)]['g'], search=(0.5, None))
    print(f"{A+'-'+B:<8}{m['peak_r']:>9.3f}{m['height']:>9.3f}"
          f"{m['fwhm']:>9.3f}{m['r2']:>8.3f}{m['rmse']:>9.4f}")

## 5. 결합 cutoff 비교 — 고정값 vs g(r) 첫 최소

고정값(LAMMPS와 동일)과 데이터에서 도출한 첫 최소값을 함께 보여줍니다.
둘이 크게 다르면 cutoff 선택을 재검토하세요. (이후 분석은 고정값 `CM` 사용)

In [ ]:
derived = cut.derive_cutoffs(R, pairs, fallback=CM)
rows = cut.compare(CM, derived, [p for p in pairs if CM.get(*p)>0])
print(cut.format_table(rows))

## 6. R(r) = 4πr²ρ·g(r)  와  reduced G(r) = 4πrρ·(g−1)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
rho0 = R['rho0']
for (A,B) in pairs:
    axes[0].plot(R['r'], rdf.R_of_r(R['r'], R[(A,B)]['g'], rho0), lw=1.1, label=f'{A}-{B}')
axes[0].plot(R['r'], rdf.R_of_r(R['r'], R['total']['g'], rho0), 'k-', lw=1.6, label='total')
axes[0].set_xlabel('r (Å)'); axes[0].set_ylabel('R(r)'); axes[0].set_title('R(r)=4πr²ρg(r)')
axes[0].legend(ncol=2, fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(R['r'], rdf.G_of_r(R['r'], R['total']['g'], rho0), 'k-', lw=1.4)
axes[1].axhline(0, color='gray', lw=0.6)
axes[1].set_xlabel('r (Å)'); axes[1].set_ylabel('G(r)'); axes[1].set_title('reduced G(r) (total)')
axes[1].grid(alpha=0.3); plt.show()

## 7. 배위수 CN  (cutoff 기반, 시간평균 ±1σ)

각 중심 원소 A에 대한 이웃 원소 B의 평균 수, 총 CN, 그리고 CN 분포.

In [ ]:
CN = coordination.coordination_numbers(traj, CM, n_blocks=NBLOCKS)
for A in species:
    parts = ', '.join(f'{B}:{CN["cn"][A][B][0]:.3f}±{CN["cn"][A][B][1]:.3f}' for B in species if CM.get(A,B)>0)
    tot = CN['cn_total'][A]
    print(f'{A}: total CN = {tot[0]:.3f} ± {tot[1]:.3f}   [{parts}]')

fig, ax = plt.subplots(figsize=(8,4.5))
w = 0.8/len(species)
for i,A in enumerate(species):
    mean,err = CN['distribution'][A]
    ax.bar(CN['cn_bins']+i*w, mean, w, yerr=err, label=A, color=colors[A], alpha=0.8, capsize=2)
ax.set_xlabel('coordination number'); ax.set_ylabel('P(CN)')
ax.set_title('Coordination-number distribution'); ax.legend(); ax.grid(alpha=0.3, axis='y'); plt.show()

## 8. ADF — 결합각 분포  (B–A–C, 중심 A)

결합 가능한 모든 삼중항을 자동 생성합니다 (SiCN이면 preset 삼중항 사용 가능).

In [ ]:
# 결합 가능한 삼중항 자동 생성 (A 중심, A-B/A-C cutoff>0)
triplets = []
for A in species:
    bonded = [B for B in species if CM.get(A,B) > 0]
    for i,B in enumerate(bonded):
        for C in bonded[i:]:
            triplets.append((B,A,C))
ADF = coordination.adf(traj, triplets, CM, nbins=180, n_blocks=NBLOCKS)

fig, ax = plt.subplots(figsize=(9,5.5))
for t in triplets:
    p = ADF[t]['p']
    if np.all(np.isnan(p)): continue
    ax.plot(ADF['theta'], p, lw=1.2, label=f'{t[0]}-{t[1]}-{t[2]}')
for ref,lab in [(109.47,'109.5° tetra'),(120,'120° planar')]:
    ax.axvline(ref, color='gray', ls='--', lw=0.8)
ax.set_xlabel('angle (deg)'); ax.set_ylabel('P(θ)'); ax.set_xlim(0,180)
ax.set_title('Angular distribution function'); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3); plt.show()

## 9. CSRO — Warren-Cowley α$_{AB}$

α<0 결합 선호(클러스터링), α>0 회피, α=0 무작위.

In [ ]:
WC = csro.warren_cowley(traj, CM, n_blocks=NBLOCKS)
sp = WC['species']; A = WC['alpha']['mean']
fig, ax = plt.subplots(figsize=(5.5,4.8))
vmax = np.nanmax(np.abs(A)); im = ax.imshow(A, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(sp))); ax.set_yticks(range(len(sp)))
ax.set_xticklabels(sp); ax.set_yticklabels(sp)
ax.set_xlabel('B (neighbor)'); ax.set_ylabel('A (central)')
for i in range(len(sp)):
    for j in range(len(sp)):
        if np.isnan(A[i,j]): continue
        ax.text(j,i,f'{A[i,j]:+.3f}\n±{WC["alpha"]["err"][i,j]:.3f}', ha='center', va='center', fontsize=9)
ax.set_title('Warren-Cowley α (blue=clustering, red=avoidance)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 10. Voronoi — 셀 부피 · 면 수 (기하 배위수)

In [ ]:
VO = voronoi.voronoi(traj, n_blocks=NBLOCKS)
for A in species:
    print(f'{A}: V = {VO["volume"][A][0]:.3f} ± {VO["volume"][A][1]:.3f} Å³   '
          f'faces = {VO["faces"][A][0]:.2f} ± {VO["faces"][A][1]:.2f}')
lf = VO['last_frame']
fig, axes = plt.subplots(1,2, figsize=(13,4.5))
for A in species:
    m = lf['elements']==A
    axes[0].hist(lf['volumes'][m], bins=40, alpha=0.55, label=A, color=colors[A])
    axes[1].hist(lf['faces'][m], bins=np.arange(-0.5, lf['faces'].max()+1.5), alpha=0.55, label=A, color=colors[A])
axes[0].set_xlabel('Voronoi cell volume (Å³)'); axes[0].set_ylabel('count'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('face count (geometric CN)'); axes[1].set_ylabel('count'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Voronoi (last frame distributions)'); plt.show()

## 11. BOO — Steinhardt Q4 · Q6 · W4 · W6

참고: 사면체/다이아몬드 Q4≈0.51 Q6≈0.63, FCC Q4≈0.19 Q6≈0.58, 무작위→0.

In [ ]:
BO = boo.steinhardt(traj, num_neighbors=BOO_K, n_blocks=NBLOCKS)
for A in species:
    print(f'{A}: Q4={BO["Q4"][A][0]:.3f}±{BO["Q4"][A][1]:.3f}  '
          f'Q6={BO["Q6"][A][0]:.3f}±{BO["Q6"][A][1]:.3f}')
lf = BO['last_frame']
fig, axes = plt.subplots(1,2, figsize=(13,4.8))
for A in species:
    m = lf['elements']==A
    axes[0].scatter(lf['Q4'][m], lf['Q6'][m], s=4, alpha=0.3, label=A, color=colors[A])
    axes[1].hist(lf['Q6'][m], bins=50, density=True, alpha=0.55, label=A, color=colors[A])
axes[0].scatter([0.510],[0.628], marker='*', s=200, c='gold', edgecolor='k', label='tetra', zorder=9)
axes[0].set_xlabel('Q4'); axes[0].set_ylabel('Q6'); axes[0].set_title(f'Q4–Q6 map (k={BOO_K})'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].axvline(0.628, color='purple', ls='--', lw=1, label='tetra 0.63')
axes[1].set_xlabel('Q6'); axes[1].set_ylabel('density'); axes[1].set_title('Q6 distribution'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.show()

## 12. 혼성화 분류 — CN + 평면성(ω)

CN=3 원자의 면외각 ω로 평면(sp²)/피라미드(sp³)를 구분합니다.

In [ ]:
HY = hybridization.classify(traj, CM, planar_thresh=15.0, n_blocks=NBLOCKS)
classes = HY['classes']
fig, axes = plt.subplots(1,2, figsize=(14,4.8))
x = np.arange(len(classes)); w = 0.8/len(species)
for i,A in enumerate(species):
    mean,err = HY['fractions'][A]
    axes[0].bar(x+i*w, mean, w, yerr=err, label=A, color=colors[A], alpha=0.8, capsize=2)
axes[0].set_xticks(x+0.4-w/2); axes[0].set_xticklabels(classes, rotation=20, fontsize=9)
axes[0].set_ylabel('fraction'); axes[0].set_title('Hybridization classes'); axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')
for A in species:
    om = HY['omega_last'][A]
    if len(om)>0: axes[1].hist(om, bins=40, range=(0,60), alpha=0.55, label=f'{A} (CN3)', color=colors[A])
axes[1].axvline(15, color='red', ls='--', label='planar thr 15°'); axes[1].axvline(35.26, color='orange', ls='--', label='tetra 35.3°')
axes[1].set_xlabel('ω (deg)'); axes[1].set_ylabel('count'); axes[1].set_title('Improper angle (CN=3, last frame)'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.show()

## 13. 사면체 질서 q (Chau–Hardwick)

q=1 완전 사면체, q≈0 무작위. 4-최근접 기하 정의.

In [ ]:
TQ = tetrahedra.tetrahedral_order(traj, n_blocks=NBLOCKS)
for A in species:
    print(f'{A}: q = {TQ["q"][A][0]:.4f} ± {TQ["q"][A][1]:.4f}')
lf = TQ['last_frame']
fig, ax = plt.subplots(figsize=(8,4.5))
for A in species:
    m = lf['elements']==A
    ax.hist(lf['q'][m], bins=50, range=(-0.5,1.0), density=True, alpha=0.55, label=A, color=colors[A])
ax.axvline(1.0, color='purple', ls='--', label='perfect tetra')
ax.set_xlabel('tetrahedral order q'); ax.set_ylabel('density'); ax.set_title('Tetrahedral order distribution')
ax.legend(); ax.grid(alpha=0.3); plt.show()

---
분석 완료. 결과는 위 출력/그림으로 확인하세요. 
다른 온도/조성은 *Config*의 `TRAJ`만 바꿔 다시 실행하면 됩니다.